<a href="https://colab.research.google.com/github/moizr1732/flyrank-ml-internship-starter/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two signals checked first, then my rule in plain words

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring (starter dataset, `data/raw/content_refresh_anonymized.csv`, 30,000 rows / 32 clients).

**Signal A — staleness (`freshness_tier`), behind FlyRank's refresh flags.** Claim: content that hasn't been touched in a while underperforms content that was recently updated. Test: weighted CTR (`total_clicks_90d / total_impressions_90d`, not the mean of per-row CTRs) per `freshness_tier` bucket, n printed per bucket.

**Signal B — CTR-vs-position (`position_tier`), behind FlyRank's CTR-fix logic.** Claim: pages ranking better (lower `avg_position`) earn a higher CTR — the whole premise of "this page ranks fine but isn't converting the clicks it should." Test: weighted CTR per `position_tier`, with the 1,205 `avg_position == 0` ("no position data") rows dropped first — those rows get miscategorized into the `top_3` tier by the raw column and would fake the result if left in.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/moizr1732/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv")
print("rows, cols:", df.shape)

# ---- Signal A: staleness (freshness_tier) vs weighted CTR ----
tier_order_fresh = ["0-30", "31-90", "91-180", "181+"]
sig_a = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    total_clicks=("clicks_90d", "sum"),
    total_impressions=("impressions_90d", "sum"),
).reindex(tier_order_fresh)
sig_a["weighted_ctr_pct"] = (sig_a["total_clicks"] / sig_a["total_impressions"] * 100).round(3)
print("\nSIGNAL A — freshness_tier vs weighted CTR (n printed)")
print(sig_a[["n", "weighted_ctr_pct"]])
print(
    "\nVerdict A: MIXED. The freshest bucket (0-30) has the highest weighted CTR (0.327%) and the\n"
    "stalest bucket (181+) is lower (0.228%), so the direction the refresh flags assume — 'stale\n"
    "content underperforms fresh content' — mostly holds. But it is NOT monotonic: the 31-90 bucket\n"
    "comes in lowest of all (0.149%) despite being 'fresher' than 91-180 and 181+, and it only has\n"
    "n=175 rows (above the ~50 floor, but the thinnest bucket by far). A clean staleness->CTR\n"
    "story does not survive fully intact, so the rule below treats staleness as a coarse two-way\n"
    "gate (>=91 days, where the pattern is clean and n is large) rather than trusting every tier order."
)

# ---- Signal B: position (position_tier) vs weighted CTR, no-data rows dropped ----
n_no_position_data = (df["avg_position"] == 0).sum()
valid_pos = df[df["avg_position"] > 0].copy()
tier_order_pos = ["top_3", "page_1", "striking", "page_3_5", "deep"]
sig_b = valid_pos.groupby("position_tier").agg(
    n=("content_id", "count"),
    total_clicks=("clicks_90d", "sum"),
    total_impressions=("impressions_90d", "sum"),
).reindex(tier_order_pos)
sig_b["weighted_ctr_pct"] = (sig_b["total_clicks"] / sig_b["total_impressions"] * 100).round(3)
print(f"\nDropped {n_no_position_data} rows with avg_position == 0 ('no data', not position zero) —")
print("the raw position_tier column wrongly files every one of them under top_3 (<=3 satisfies 0<=3).")
print("\nSIGNAL B — position_tier vs weighted CTR, no-data rows removed (n printed)")
print(sig_b[["n", "weighted_ctr_pct"]])
print(
    "\nVerdict B: CONFIRMED. Weighted CTR falls in step with position: top_3 0.489% -> page_1\n"
    "0.350% -> striking 0.347% -> page_3_5 0.155% -> deep 0.041%. Every bucket clears n>1,000. The\n"
    "CTR-fix premise holds: a page's own tier sets a real, data-backed expectation for its CTR, so a\n"
    "page sitting well below its own tier's weighted CTR is a legitimate 'leaving clicks on the\n"
    "table' signal, not noise."
)


rows, cols: (30000, 44)

SIGNAL A — freshness_tier vs weighted CTR (n printed)
                    n  weighted_ctr_pct
freshness_tier                         
0-30            20480             0.327
31-90             175             0.149
91-180           9171             0.291
181+              174             0.228

Verdict A: MIXED. The freshest bucket (0-30) has the highest weighted CTR (0.327%) and the
stalest bucket (181+) is lower (0.228%), so the direction the refresh flags assume — 'stale
content underperforms fresh content' — mostly holds. But it is NOT monotonic: the 31-90 bucket
comes in lowest of all (0.149%) despite being 'fresher' than 91-180 and 181+, and it only has
n=175 rows (above the ~50 floor, but the thinnest bucket by far). A clean staleness->CTR
story does not survive fully intact, so the rule below treats staleness as a coarse two-way
gate (>=91 days, where the pattern is clean and n is large) rather than trusting every tier order.

Dropped 1205 rows with avg_

**The rule, in plain words:** A page earns a refresh-and-CTR review this week if it hasn't been
touched in 91+ days (the staleness cutoff where Signal A's pattern is clean and well-sampled), it
still pulls real search volume (so the team isn't spending review time on noise), and its own CTR
sits below what pages at its own position tier typically earn (Signal B). Score ranks the flagged
pages by how much impression volume is riding on that gap, so the biggest opportunities surface
first.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*


In [2]:
STALE_DAYS_CUTOFF = 91      # from Signal A: the boundary where the staleness pattern is clean and n is large
VOLUME_FLOOR = 500          # impressions_90d floor so the queue isn't ranking noise

# expected CTR per position tier, straight from Signal B's confirmed bucket table
tier_expected_ctr = sig_b["weighted_ctr_pct"].to_dict()

stale = (df["days_since_last_update"] >= STALE_DAYS_CUTOFF).astype(int)
visible = (df["impressions_90d"] >= VOLUME_FLOOR).astype(int)

df["expected_ctr_for_tier"] = df["position_tier"].map(tier_expected_ctr)
has_position_data = df["avg_position"] > 0
underperforming = ((df["ctr"] < df["expected_ctr_for_tier"]) & has_position_data).astype(int)

# readable on purpose: three plain gates, multiplied by the volume that makes the opportunity worth it
df["score"] = stale * visible * underperforming * df["impressions_90d"]

df["reason_code"] = np.where(df["score"] > 0, "stale_visible_ctr_underperforming", "none")
df["action_label"] = np.where(df["score"] > 0, "review_for_refresh_and_ctr", "no_action_needed")

n_flagged = int((df["score"] > 0).sum())
print(f"Flagged for review: {n_flagged} of {len(df)} rows ({n_flagged / len(df):.1%})")

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

out_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action_label",
    "days_since_last_update", "freshness_tier", "impressions_90d", "impression_tier",
    "ctr", "avg_position", "position_tier", "expected_ctr_for_tier",
]

import os
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)


Flagged for review: 4563 of 30000 rows (15.2%)
Wrote work/outputs/baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action_label,days_since_last_update,freshness_tier,impressions_90d,impression_tier,ctr,avg_position,position_tier,expected_ctr_for_tier
0,1,content_5fe46e04994d,client_4e07408562,517715,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,517715,excellent,0.14,4.2,page_1,0.350
1,2,content_cb112fce36be,client_19581e27de,309910,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,309910,excellent,0.16,5.6,page_1,0.350
2,3,content_36ff89c8214e,client_19581e27de,295097,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,295097,excellent,0.05,7.3,page_1,0.350
3,4,content_b28d1efd668f,client_6208ef0f77,286608,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,286608,excellent,0.06,26.2,page_3_5,0.155
4,5,content_813e88069237,client_6208ef0f77,233561,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,233561,excellent,0.06,26.2,page_3_5,0.155
5,6,content_c8e9d6ab9013,client_19581e27de,208678,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,208678,excellent,0.00,9.7,page_1,0.350
6,7,content_b511d4bc4ad2,client_6208ef0f77,205915,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,205915,excellent,0.14,27.9,page_3_5,0.155
7,8,content_d17681677e69,client_19581e27de,201584,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,201584,excellent,0.24,5.8,page_1,0.350
8,9,content_a7427266c305,client_19581e27de,201111,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,201111,excellent,0.11,5.7,page_1,0.350
9,10,content_c5063073d048,client_6208ef0f77,192205,stale_visible_ctr_underperforming,review_for_refresh_and_ctr,104,91-180,192205,excellent,0.24,12.5,striking,0.347


## 3. Top-10 review

*For each of the top ten: the action, why it's there, and what would make it wrong.*


In [3]:
top10 = queue.head(10)
for _, r in top10.iterrows():
    print(f"#{r['rank']} {r['content_id']} (client {r['client_id']})")
    print(f"  action: {r['action_label']}")
    print(
        f"  why: not updated in {r['days_since_last_update']}d (freshness_tier {r['freshness_tier']}), "
        f"{r['impressions_90d']:,} impressions_90d ({r['impression_tier']}), ctr {r['ctr']}% vs "
        f"{r['expected_ctr_for_tier']}% expected for {r['position_tier']} (avg_position {r['avg_position']})"
    )
    print(
        "  what would make it wrong: if this page's true intent is already served without a click "
        "(e.g. a featured-snippet/knowledge-panel-eating query, or a navigational/branded query where "
        "users recognize the result and don't need to click), the CTR gap is a SERP-feature or intent "
        "artifact, not a fixable title/meta problem — a manual SERP check before touching the page "
        "would catch this."
    )
    print()


#1 content_5fe46e04994d (client client_4e07408562)
  action: review_for_refresh_and_ctr
  why: not updated in 104d (freshness_tier 91-180), 517,715 impressions_90d (excellent), ctr 0.14% vs 0.35% expected for page_1 (avg_position 4.2)
  what would make it wrong: if this page's true intent is already served without a click (e.g. a featured-snippet/knowledge-panel-eating query, or a navigational/branded query where users recognize the result and don't need to click), the CTR gap is a SERP-feature or intent artifact, not a fixable title/meta problem — a manual SERP check before touching the page would catch this.

#2 content_cb112fce36be (client client_19581e27de)
  action: review_for_refresh_and_ctr
  why: not updated in 104d (freshness_tier 91-180), 309,910 impressions_90d (excellent), ctr 0.16% vs 0.35% expected for page_1 (avg_position 5.6)
  what would make it wrong: if this page's true intent is already served without a click (e.g. a featured-snippet/knowledge-panel-eating query, or

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


In [4]:
print("Weak pick — rank 4 & 5, content_b28d1efd668f / content_813e88069237 (client_6208ef0f77):")
print(
    "  Both sit in position_tier 'page_3_5' at avg_position ~26, a spot where informational SERPs\n"
    "  routinely carry People-Also-Ask boxes and image packs that siphon clicks regardless of title\n"
    "  quality. The rule can't see SERP layout, only avg_position and ctr, so it can mistake a\n"
    "  crowded-SERP page for a fixable-CTR page. Both also share client_6208ef0f77 and an identical\n"
    "  days_since_last_update (104) with several other top-20 rows — worth checking that client's\n"
    "  refresh isn't just a single bulk-update event dominating the queue."
)

# ---- leakage check ----
label_bearing_cols = {"trend_direction", "trend_pct", "is_declining_label"}
used_in_score = {"days_since_last_update", "impressions_90d", "ctr", "position_tier", "avg_position"}
print("\nColumns used in score:", sorted(used_in_score))
print("Label-bearing columns present anywhere in score logic?", bool(used_in_score & label_bearing_cols))
print("impressions_last_30d / impressions_prev_30d (trend inputs) used?", False)
print(
    "No future-window or label-derived columns feed the score — every input (days_since_last_update,\n"
    "impressions_90d, ctr, avg_position, position_tier) is an observed, trailing-90-day, present-day\n"
    "signal. trend_direction / trend_pct / is_declining_label are never touched."
)


Weak pick — rank 4 & 5, content_b28d1efd668f / content_813e88069237 (client_6208ef0f77):
  Both sit in position_tier 'page_3_5' at avg_position ~26, a spot where informational SERPs
  routinely carry People-Also-Ask boxes and image packs that siphon clicks regardless of title
  quality. The rule can't see SERP layout, only avg_position and ctr, so it can mistake a
  crowded-SERP page for a fixable-CTR page. Both also share client_6208ef0f77 and an identical
  days_since_last_update (104) with several other top-20 rows — worth checking that client's
  refresh isn't just a single bulk-update event dominating the queue.

Columns used in score: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d', 'position_tier']
Label-bearing columns present anywhere in score logic? False
impressions_last_30d / impressions_prev_30d (trend inputs) used? False
No future-window or label-derived columns feed the score — every input (days_since_last_update,
impressions_90d, ctr, avg_position, p

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
